In [34]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import requests
from bs4 import BeautifulSoup

In [35]:
import time
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Tarayıcı oturumunu başlatıyoruz
driver = uc.Chrome(version_main=148)
tum_ilan_linkleri = []

try:
    # 1. sayfadan 17. sayfaya kadar tüm sayfaları dolaşır (18 dahil değildir)
    for sayfa in range(1, 18):
        url = f"https://www.emlakjet.com/kiralik-konut/konya/{sayfa}/"
        print(f"🔄 Sayfa {sayfa}/17 dolaşılıyor: {url}")
        
        driver.get(url)
        
        # Sayfanın temel yapısının yüklenmesini bekle
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
        except Exception:
            print(f"⚠️ Sayfa {sayfa} yüklenirken zaman aşımı oldu, sonraki sayfaya geçiliyor...")
            continue
            
        # Cloudflare ve sayfa içeriğinin tam oturması için kısa bir bekleme
        time.sleep(3)
        
        # Sayfayı yavaşça aşağı kaydırarak dinamik ilanları tetikle
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        # İçeriğinde '/ilan/' kelimesi geçen tüm ilan linklerini JavaScript ile süz
        sayfa_linkleri = driver.execute_script("""
            let links = [];
            let anchors = document.querySelectorAll("a[href*='/ilan/']");
            anchors.forEach(a => {
                if(a.href && !links.includes(a.href)) {
                    links.push(a.href);
                }
            });
            return links;
        """)
        
        tum_ilan_linkleri.extend(sayfa_linkleri)
        print(f"✅ Sayfa {sayfa} tamamlandı. Bu sayfadan {len(sayfa_linkleri)} adet link alındı.")
        time.sleep(1)

finally:
    # Tarayıcıyı kapat
    driver.quit()
    print("🤖 Tarayıcı kapatıldı, veri işleme adımına geçiliyor...")

# 1. Tekrar eden bağlantıları temizle
temiz_linkler = list(set(tum_ilan_linkleri))
print(f"\n🎉 Toplam {len(temiz_linkler)} adet benzersiz ilan linki toplandı.")


🔄 Sayfa 1/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/1/
✅ Sayfa 1 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 2/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/2/
✅ Sayfa 2 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 3/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/3/
✅ Sayfa 3 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 4/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/4/
✅ Sayfa 4 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 5/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/5/
✅ Sayfa 5 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 6/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/6/
✅ Sayfa 6 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 7/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/7/
✅ Sayfa 7 tamamlandı. Bu sayfadan 30 adet link alındı.
🔄 Sayfa 8/17 dolaşılıyor: https://www.emlakjet.com/kiralik-konut/konya/8/
✅ Sayfa 8 tamamlandı. B

In [43]:
headers = {'User-Agent': 'Mozilla/5.0'}

base_url='https://www.emlakjet.com/kiralik-konut/konya?sayfa='
page= [f"{base_url}{i}" for i in range(1,18)]

print(page)

['https://www.emlakjet.com/kiralik-konut/konya?sayfa=1', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=2', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=3', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=4', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=5', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=6', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=7', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=8', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=9', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=10', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=11', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=12', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=13', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=14', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=15', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=16', 'https://www.emlakjet.com/kiralik-konut/konya?sayfa=17']


In [44]:
house = []

for url in page:
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # ilan listesi sınıf adı
        listings = soup.find_all('div', class_='styles_listingWrapper__gVjYi')
        
        # Her bir ilanı tek tek gezmek için bu satırı ekleyin
        for listing in listings:
            # Reklam veya boş kartları atlamak için kontrol
            if listing.find('div', class_='styles_saveSeasrchWrapper__t24Cm'):
                continue
                
            link_tag = listing.find('a', href=True)
            if link_tag:
                full_link = 'https://www.emlakjet.com' + link_tag['href']
                house.append(full_link)

# Toplanan link sayısını kontrol etmek için
print(f"✅ Toplam {len(house)} adet ilan linki başarıyla toplandı!")


✅ Toplam 475 adet ilan linki başarıyla toplandı!


In [45]:
house

['https://www.emlakjet.com/ilan/karaca-dan-erenler-mahallesi-31-kiralik-ara-kat-daire-19422809',
 'https://www.emlakjet.com/ilan/isiklar-mah-yegenoglu-cad-31-kombili-kiralik-daire-19422509',
 'https://www.emlakjet.com/ilan/babi-emlaktan-kiralik-31-daire-site-ici-bakimli-19421287',
 'https://www.emlakjet.com/ilan/zirve-emlak-kiraliyorr-yarenler-31-180m2-4-kat-asansorlu-daire-19421112',
 'https://www.emlakjet.com/ilan/kucukkum-kopru-cd-sehir-hastanesi-yakini-kiralik-21-daire-19421074',
 'https://www.emlakjet.com/ilan/ugur-emlak-tan-mesaj-cad-uzeri-kismi-esyali-31-kiralik-daire-19419955',
 'https://www.emlakjet.com/ilan/joy-gayrimenkul-den-kiralik-lux-daire-19419435',
 'https://www.emlakjet.com/ilan/yenikent-prestij-konutlarinda-kiralik-bakimli-41-daire-19418268',
 'https://www.emlakjet.com/ilan/ozel-anadolu-gelisim-okullari-civari-21-esyali-daire-19418254',
 'https://www.emlakjet.com/ilan/sehir-hastanesi-yakini-esyali-kiralik-studyo-daire-0-yas-19417775',
 'https://www.emlakjet.com/ilan/

In [37]:
#df_links = pd.DataFrame(house, columns=['Ilan_Linki'])

In [38]:
#df_links.to_csv('emlakjet_ilan_linkleri.csv', index=False, encoding='utf-8-sig')

In [46]:
import pandas as pd
import numpy as np

house_data = []

for house_url in house:
    response = requests.get(house_url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        house_info = {}
        
        price_span = soup.find('span', class_='styles_price__6wmxk')
        price = price_span.get_text(strip=True) if price_span else np.nan
        
        location_span = soup.find('span', class_='styles_location__HguIg')
        if location_span:
            location_text = location_span.get_text(strip=True)
            location_parts = location_text.split('-')
            location = '-'.join(location_parts[:2]) if len(location_parts) >= 2 else np.nan
        else:
            location = np.nan
        features_div = soup.find('div', class_='styles_inner__qKPCB')
        if features_div:
            li_tags = features_div.find_all('li')
            for li in li_tags:
                spans = li.find_all('span')
                if len(spans) == 2:
                    key = spans[0].get_text(strip=True)
                    value = spans[1].get_text(strip=True)
                    house_info[key] = value


            house_info['Fiyat'] = price
            house_info['Location'] = location
            house_info['İlan linki'] = house_url
            
            house_data.append(house_info)

In [47]:
df=pd.DataFrame(house_data)

In [49]:
df

,İlan Numarası,İlan Güncelleme Tarihi,Türü,Kategorisi,Tipi,Net Metrekare,Brüt Metrekare,Oda Sayısı,Binanın Yaşı,Bulunduğu Kat,...,Aidat,WC Sayısı,Yapı Durumu,Yapı Tipi,Banyo Metrekare,Balkon Metrekare,Görüntülü Gezilebilir mi?,Enerji Kimlik Belgesi,Salon Metrekare,WC Metrekare
0,19422809,30 Mayıs 2026,Konut,Kiralık,Daire,110 m²,155 m²,3+1,3,2.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19422509,30 Mayıs 2026,Konut,Kiralık,Daire,130 m²,155 m²,3+1,21 Ve Üzeri,2.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19421287,29 Mayıs 2026,Konut,Kiralık,Daire,145 m²,165 m²,3+1,16-20,6.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19421112,29 Mayıs 2026,Konut,Kiralık,Daire,145 m²,180 m²,3+1,11-15,4.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19421074,29 Mayıs 2026,Konut,Kiralık,Daire,90 m²,100 m²,2+1,3,3.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,15484289,25 Mayıs 2026,Konut,Kiralık,Daire,70 m²,89 m²,2+1,0 (Yeni),1.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
471,14949061,22 Mayıs 2026,Konut,Kiralık,Daire,135 m²,145 m²,3+1,21 Ve Üzeri,2.Kat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
472,14829824,27 Mayıs 2026,Konut,Kiralık,Daire,140 m²,145 m²,3+1,21 Ve Üzeri,2.Kat,...,200 TL,2,İkinci El,Yığma,NaN,NaN,Evet,NaN,NaN,NaN
473,10171744,30 Mayıs 2026,Konut,Kiralık,Daire,130 m²,150 m²,3+1,21 Ve Üzeri,3.Kat,...,400 TL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
df.to_csv('house_data.csv',index=False)